In [58]:
# Import some useful modules.
import jax
import jax.numpy as jnp
import os

# Import JAX-FEM specific modules.
from jax_fem.problem import Problem
from jax_fem.solver import solver, dynamic_relax_solve
from jax_fem.utils import save_sol
from jax_fem.generate_mesh import rectangle_mesh, get_meshio_cell_type, Mesh

In [68]:
# Define constitutive relationship.
class HyperElasticity(Problem):
    # The function 'get_tensor_map' overrides base class method. Generally, JAX-FEM
    # solves -div(f(u_grad)) = b. Here, we define f(u_grad) = P. Notice how we first
    # define 'psi' (representing W), and then use automatic differentiation (jax.grad)
    # to obtain the 'P_fn' function.
    def get_tensor_map(self):

        def psi(F_2d):
            F = jnp.array([[F_2d[0, 0], F_2d[0, 1], 0.], 
                          [F_2d[1, 0], F_2d[1, 1], 0.],
                          [0., 0., 1.]])
            E = 10.
            nu = 0.3
            mu = E / (2. * (1. + nu))
            kappa = E / (3. * (1. - 2. * nu))
            
            J = jnp.linalg.det(F)
            J_safe = jnp.clip(J, 1.0e-8, 1.0e8)
            Jinv = J_safe**(-2. / 3.)
            I1 = jnp.trace(F.T @ F)
            energy = (1.0 / 2.) * (Jinv * I1 - 2.) + (3.0 / 2.) * (J_safe - 1.)**2.
            return energy

        P_fn = jax.grad(psi)

        def first_PK_stress(u_grad):
            I = jnp.eye(self.dim)
            F = u_grad + I
            P = P_fn(F)
            return P

        return first_PK_stress

In [69]:

import numpy as np
from dolfinx.io.gmsh import read_from_msh
from mpi4py import MPI

In [70]:
msh_data = read_from_msh("mesh_with_hole.msh", MPI.COMM_WORLD, 0, 2)  # 2D mesh
domain = msh_data.mesh

# 2. Extract node coordinates (geometry points)
# shape (num_nodes, gdim)
node_coords = domain.geometry.x  # numpy array
num_nodes, gdim = node_coords.shape
print(f"Nodes: {num_nodes}, dimension: {gdim}")

# 3. Extract cell connectivity (triangles)
tdim = domain.topology.dim
domain.topology.create_connectivity(tdim, 0)
cells = domain.geometry.dofmap

Info    : Reading 'mesh_with_hole.msh'...
Info    : 13 entities
Info    : 1220 nodes
Info    : 2224 elements
Info    : Done reading 'mesh_with_hole.msh'
Nodes: 1220, dimension: 3


In [71]:
# Specify mesh-related information (first-order hexahedron element).
ele_type = 'TRI3'
cell_type = get_meshio_cell_type(ele_type)
Lx, Ly, Lz = 1., 1., 1.
# mesh = rectangle_mesh(Nx=20,
#                        Ny=20,
#                        domain_x=Lx,
#                        domain_y=Ly)
mesh = Mesh(node_coords[:, :2], cells)

In [72]:
# Define boundary locations.
def left(point):
    return jnp.isclose(point[0], 0., atol=1e-5)


def right(point):
    return jnp.isclose(point[0], Lx, atol=1e-5)

def bottom(point):
    return jnp.isclose(point[1], 0., atol=1e-5)


def top(point):
    return jnp.isclose(point[1], Ly, atol=1e-5)
# Define value function.
def zero_dirichlet_val(point):
    return 0.


def dirichlet_val_x2(point):
    return 1.0



dirichlet_bc_info = [
    [left] * 2 + [right] * 2, 
    [0, 1] * 2,
    [zero_dirichlet_val] * 2 + [zero_dirichlet_val, dirichlet_val_x2]
                     ]

In [73]:
# Create an instance of the problem.
problem = HyperElasticity(mesh,
                          vec=2,
                          dim=2,
                          ele_type=ele_type,
                          dirichlet_bc_info=dirichlet_bc_info)

[11-24 14:09:39][DEBUG] jax_fem: Computing shape function values, gradients, etc.
[11-24 14:09:39][DEBUG] jax_fem: ele_type = TRI3, quad_points.shape = (num_quads, dim) = (1, 2)
[11-24 14:09:39][DEBUG] jax_fem: face_quad_points.shape = (num_faces, num_face_quads, dim) = (3, 1, 2)
[11-24 14:09:39][DEBUG] jax_fem: Done pre-computations, took 0.02913832664489746 [s]
[11-24 14:09:39][INFO] jax_fem: Solving a problem with 2224 cells, 1220x2 = 2440 dofs.
[11-24 14:09:39][INFO] jax_fem: Element type is TRI3, using 1 quad points per element.


In [75]:
# Solve the defined problem.
petsc_options = {
        "snes_type": "newtonls",
        "snes_linesearch_type": "bt",
        "snes_monitor": None,
        "snes_atol": 1e-6,
        "snes_rtol": 1e-6,
        "snes_stol": 1e-6,
        "ksp_type": "preonly",
        "pc_type": "lu",
        "pc_factor_mat_solver_type": "mumps",}
sol_list = solver(problem, solver_options={'petsc_solver': petsc_options})

# sol_list = dynamic_relax_solve(problem, 1e-6)

[11-24 14:09:53][DEBUG] jax_fem: Calling the row elimination solver for imposing Dirichlet B.C.
[11-24 14:09:53][DEBUG] jax_fem: Start timing
[11-24 14:09:53][DEBUG] jax_fem: Computing cell Jacobian and cell residual...
[11-24 14:09:53][DEBUG] jax_fem: Function split_and_compute_cell took 0.0089 seconds
[11-24 14:09:53][DEBUG] jax_fem: Creating sparse matrix with scipy...
[11-24 14:09:53][DEBUG] jax_fem: Before, l_2 res = 5.916079783099616, relative l_2 res = 1.0
[11-24 14:09:53][DEBUG] jax_fem: Solving linear system...
[11-24 14:09:53][DEBUG] jax_fem: PETSc Solver - Solving linear system with ksp_type = preonly, pc = lu
[11-24 14:09:53][DEBUG] jax_fem: PETSc Solver - Finished solving, linear solve res = 4.377545950001754e-14
[11-24 14:09:53][DEBUG] jax_fem: Computing cell Jacobian and cell residual...
[11-24 14:09:53][DEBUG] jax_fem: Function split_and_compute_cell took 0.0084 seconds
[11-24 14:09:53][DEBUG] jax_fem: Creating sparse matrix with scipy...
[11-24 14:09:53][DEBUG] jax_fem

AssertionError: PETSc linear solver failed to converge, err = 1553.8079719011691

In [ ]:
# Store the solution to local file.
vtk_path = os.path.join(data_dir, f'vtk/u.vtu')
save_sol(problem.fes[0], sol_list[0], vtk_path)